In [1]:
import pandas as pd
import numpy as np
import xgboost as xgb
import optuna
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.metrics import average_precision_score

# 1. LOAD THE ACTUAL KAGGLE DATA
def load_real_data(filepath='creditcard.csv'):
    try:
        df = pd.read_csv(filepath)
        print(f"Dataset loaded successfully. Shape: {df.shape}")
        return df
    except FileNotFoundError:
        print(f"Error: '{filepath}' not found. Please download it from Kaggle.")
        return None

df = load_real_data()

if df is not None:
    # Prepare features and target
    X = df.drop('Class', axis=1).values
    y = df['Class'].values

    # 2. OPTIMIZED OBJECTIVE FUNCTION
    def objective(trial):
        # Hyperparameter Search Space
        params = {
            'max_depth': trial.suggest_int('max_depth', 3, 10),
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2, log=True),
            'subsample': trial.suggest_float('subsample', 0.6, 1.0),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
            # scale_pos_weight = count(negative) / count(positive)
            # For this dataset, it's roughly 99/1 = 99.
            'scale_pos_weight': trial.suggest_float('scale_pos_weight', 50, 150),
            'n_estimators': 500,
            'eval_metric': 'aucpr',
            'use_label_encoder': False,
            'verbosity': 0
        }

        # Using a single split for hyperparameter tuning to save time
        # We use Stratified split to ensure fraud cases are in both sets
        X_train, X_val, y_train, y_val = train_test_split(
            X, y, test_size=0.2, stratify=y, random_state=42
        )

        model = xgb.XGBClassifier(**params)
        
        # Fit with early stopping to prevent overfitting and save time
        model.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            verbose=False
        )

        # Predict probabilities for the positive class (Fraud)
        y_pred = model.predict_proba(X_val)[:, 1]
        
        # Calculate Average Precision
        ap = average_precision_score(y_val, y_pred)
        
        # Report to Optuna for pruning
        trial.report(ap, 0) 
        if trial.should_prune():
            raise optuna.TrialPruned()
            
        return ap

    # 3. RUN THE OPTIMIZATION
    # We use 'maximize' because we want the highest Average Precision
    study = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=42),
        pruner=optuna.pruners.SuccessiveHalvingPruner()
    )

    print("Starting Optimization on Kaggle Data...")
    # n_trials=20 is a good starting point for this large dataset
    study.optimize(objective, n_trials=20, n_jobs=1) 

    print("\n--- Optimization Results ---")
    print(f"Best Average Precision (AP): {study.best_value:.4f}")
    print(f"Best Hyperparameters: {study.best_params}")


/opt/anaconda3/envs/tf_m1/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-08-07 23:52:37,501] A new study created in memory with name: no-name-fcdf5c3d-a969-4afc-a9e9-2fe9c9e455c1


Dataset loaded successfully. Shape: (284807, 31)
Starting Optimization on Kaggle Data...


[I 2026-08-07 23:52:40,895] Trial 0 finished with value: 0.8832839331811186 and parameters: {'max_depth': 5, 'learning_rate': 0.17254716573280354, 'subsample': 0.8927975767245621, 'colsample_bytree': 0.8394633936788146, 'scale_pos_weight': 65.60186404424365}. Best is trial 0 with value: 0.8832839331811186.
[I 2026-08-07 23:52:43,843] Trial 1 finished with value: 0.8098306653233786 and parameters: {'max_depth': 4, 'learning_rate': 0.011900590783184251, 'subsample': 0.9464704583099741, 'colsample_bytree': 0.8404460046972835, 'scale_pos_weight': 120.80725777960456}. Best is trial 0 with value: 0.8832839331811186.
[I 2026-08-07 23:52:46,778] Trial 2 finished with value: 0.8662179667634111 and parameters: {'max_depth': 3, 'learning_rate': 0.1827602783178572, 'subsample': 0.9329770563201687, 'colsample_bytree': 0.6849356442713105, 'scale_pos_weight': 68.18249672071006}. Best is trial 0 with value: 0.8832839331811186.
[I 2026-08-07 23:52:49,868] Trial 3 finished with value: 0.8618908191311971


--- Optimization Results ---
Best Average Precision (AP): 0.8848
Best Hyperparameters: {'max_depth': 5, 'learning_rate': 0.09843934055953554, 'subsample': 0.8849843275089867, 'colsample_bytree': 0.6236455099722072, 'scale_pos_weight': 123.81598936970349}
